# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")
pprint.pprint(metadata.keywords)

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

We print the record sets, their `@id`, and the fields and columns within each. All identifiers are referenced by `@id` according to the Croissant schema.

In [ ]:
from pprint import pprint

# List all record sets with their fields
record_sets = dataset.record_sets
if not record_sets:
    print('No record sets found in this dataset. Attempting to look for tabular data via files...')
    # Try to infer record sets from available distribution, if possible.
    print('schema.org/distribution objects:')
    pprint(metadata.distribution)
else:
    for rs in record_sets:
        print(f"Record Set: {rs.name} (@id: {rs['@id']})")
        print('  Fields:')
        for field in rs.fields:
            print(f"    {field.name} (@id: {field['@id']}) type={field.data_type}")
        print('  Columns:')
        for col in rs.columns:
            print(f"    {col.name} (@id: {col['@id']}) source={col.source}")

## 3. Data Extraction
Load data from a record set into a DataFrame for analysis.

We'll attempt to extract tabular records, referencing the record set(s) by their `@id`. If the dataset does not define Croissant record sets, we will attempt to load the first available CSV table.

In [ ]:
import os

dataframes = {}
loaded = False

# Main logic for Croissant-compliant datasets
record_sets = dataset.record_sets

if record_sets:
    record_set_ids = [rs['@id'] for rs in record_sets]
    print(f"Record set @ids: {record_set_ids}")
    for rec_id in record_set_ids:
        try:
            records = list(dataset.records(record_set=rec_id))
            df = pd.DataFrame(records)
            dataframes[rec_id] = df
            print(f"Loaded record set {rec_id} with columns: {df.columns.tolist()}")
            loaded = True
        except Exception as e:
            print(f"Could not load records for {rec_id}: {e}")
    if loaded:
        # Preview first record set
        first_rec_id = record_set_ids[0]
        display_columns = dataframes[first_rec_id].columns.tolist()
        print(f"\nColumns for {first_rec_id}: {display_columns}\n")
        dataframes[first_rec_id].head()

# If no Croissant record sets found, look for CSV
if not loaded:
    # Use the first CSV in distribution if present
    dist = metadata.distribution if hasattr(metadata, 'distribution') else []
    csv_urls = []
    for d in dist:
        if d.get('encodingFormat', '').lower() in ['text/csv', 'csv']:
            csv_urls.append(d.get('contentUrl', d.get('url')))
    if csv_urls:
        print(f"Found CSV file: {csv_urls[0]}")
        df = pd.read_csv(csv_urls[0])
        dataframes['main_csv'] = df
        print(df.head())
    else:
        print('No CSV distribution found.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

We select one numeric field (if present), filter records, normalize, and group by a categorical field. All fields are referenced by their Croissant `@id`.

In [ ]:
import numpy as np

# Pick DataFrame to work with
if dataframes:
    df_key = list(dataframes.keys())[0]
    df = dataframes[df_key]
    print(f"Working with DataFrame key: {df_key}")
    
    # Try to detect a numeric field (float/int)
    num_fields = [col for col in df.columns 
                 if np.issubdtype(df[col].dropna().apply(lambda v: isinstance(v, (int, float)) or (isinstance(v, str) and v.replace('.','',1).isdigit())).all(), bool)
                 or np.issubdtype(df[col].dtype, np.number)]
    if not num_fields:
        # Try fallback by heuristics
        for c in df.columns:
            try:
                df[c] = pd.to_numeric(df[c], errors='ignore')
            except:
                pass
        num_fields = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
        if not num_fields:
            print('No numeric field found for EDA.')
            num_field = None
        else:
            num_field = num_fields[0]
    else:
        num_field = num_fields[0]
    print(f"Numeric field: {num_field}")

    if num_field:
        # Drop rows where num_field is not numeric
        filtered = df[pd.to_numeric(df[num_field], errors='coerce').notna()]
        df[num_field] = pd.to_numeric(df[num_field], errors='coerce')
        threshold = filtered[num_field].mean()
        filtered_df = filtered[filtered[num_field] > threshold]
        print(f"Filtered records with {num_field} > {threshold:.2f}:")
        print(filtered_df.head())

        filtered_df[f"{num_field}_normalized"] = (
            filtered_df[num_field] - filtered_df[num_field].mean()
        ) / filtered_df[num_field].std()
        print(f"Normalized {num_field} for filtered records:")
        print(filtered_df[[num_field, f"{num_field}_normalized"]].head())

        # Try to pick group field (categorical)
        cat_fields = [col for col in df.columns if df[col].dtype == object and col != num_field]
        group_field = cat_fields[0] if cat_fields else None
        print(f'Group field: {group_field}')
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[num_field].mean().to_frame()
            grouped_df = grouped_df.rename(columns={num_field: f"avg_{num_field}"})
            print(f"Grouped data (average {num_field}) by {group_field}:")
            print(grouped_df.head())
else:
    print('No data loaded for analysis in EDA step.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll demonstrate histogram and boxplot for the selected numeric field, and a bar plot if grouped analysis was available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df_key = list(dataframes.keys())[0]
    df = dataframes[df_key]
    if num_field:
        plt.figure(figsize=(10,4))
        plt.subplot(1,2,1)
        sns.histplot(df[num_field].dropna(), kde=True, bins=30)
        plt.title(f"Histogram of {num_field}")
        
        plt.subplot(1,2,2)
        sns.boxplot(x=df[num_field].dropna())
        plt.title(f"Boxplot of {num_field}")
        plt.tight_layout()
        plt.show()
        
        if 'grouped_df' in locals() and group_field:
            grouped_df.reset_index().plot.bar(x=group_field, y=f"avg_{num_field}", legend=False)
            plt.title(f"Average {num_field} per {group_field}")
            plt.ylabel(f"Average {num_field}")
            plt.show()
    else:
        print('No numeric field available for visualization.')
else:
    print('No data to visualize.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the dataset and extracted available tabular record sets (or CSV).
- Using automated heuristics, we selected one numeric field for demonstration purposes, filtered, normalized, and visualized its distribution.
- For future analyses, consult the Croissant schema for exact `@id` for semantic references, and adjust the notebook code to use those `@id` values for repeatable and robust processing.